# L5b: Portfolio Simulation using Multiple Asset Geometric Brownian Motion
In this lecture, we take our first steps toward developing optimal portfolios, i.e., collections of equity, ETFs, and fixed income assets. Today, we'll focus on one key aspect of portfolio development, namely the simulation of portfolio performance.

> __Learning Objectives__
>
> By the end of this lecture, you will be able to:
> * **Portfolio Simulation using MAGBM**: Simulate portfolio performance using Multiple Asset Geometric Brownian Motion (MAGBM) models, including the mathematical formulation with covariance matrices and the Cholesky factorization approach for correlated asset price evolution.
> * **Modern Portfolio Theory and Optimization**: Solve the minimum variance portfolio problem as a constrained quadratic programming problem with weight sum constraints, non-negativity constraints, and minimum return requirements.
> * **Portfolio Valuation and Performance Analysis**: Use the dollar fraction allocation vector and future asset price estimates to determine portfolio performance over holding periods using a net present value approach.


This is going to be an interesting lecture, so let's get started!

___

## Examples

> [▶ Let's compute the covariance matrix for our dataset](CHEME-5660-L5a-Example-CovarianceMatrix-Fall-2026.ipynb). In this example, we compute the empirical covariance matrix for a portfolio of firms using historical growth rate data. We'll think about how to use the covariance matrix to select firms for our portfolio.

> [▶ Let's simulate a portfolio using MAGBM](CHEME-5660-L5b-Example-MAGBM-Portfolio-Fall-2026.ipynb). In this example, we simulate the price of a portfolio of assets using the MAGBM model. We'll assume we have a portfolio consisting of several stocks and ETFs, each with a specified weight in the portfolio. We'll use historical data to estimate the parameters of the MAGBM model and then simulate the portfolio's price over time using out-of-sample data.

> [▶ Let's compute the optimal weights for a portfolio using the minimum variance portfolio approach](CHEME-5660-L5b-Example-Data-MinVar-Portfolio-Fall-2026.ipynb). In this example, we compute the optimal weights for a portfolio of assets using the minimum variance portfolio approach. We'll compare the performance of this portfolio to an equally weighted portfolio and an index fund, e.g., `SPY`.
___

## Concept Review: Multiple Asset Geometric Brownian Motion (MAGBM) and Covariance
In the previous lecture, we extended the single asset geometric Brownian motion (GBM) model to describe multiple assets. The key new ingredient in the multiple asset GBM (MAGBM) model is the GBM covariance rate $\mathbf C$ which captures how different firms' returns move together.

### Covariance Matrix Example
Let's finish up our discussion of the covariance matrix by doing the last part of the example we started in the previous lecture:

> __Example__
> 
> [▶ Let's compute the covariance matrix for our dataset](CHEME-5660-L5a-Example-CovarianceMatrix-Fall-2026.ipynb). In this example, we compute the empirical covariance matrix for a portfolio of firms using historical growth rate data. We'll think about how to use the covariance matrix to select firms for our portfolio.


### MAGBM Recap
The canonical price SDE is $dS_i/S_i=\mu_i\,dt+\sum_j a_{ij}\,dW_j$, where $\mathbf A\mathbf A^{\top}=\mathbf C$. Applying Itô's lemma gives the expected log growth $\bar g_i=\mu_i-C_{ii}/2$. Therefore, the one-step ahead prediction of the price of asset $i$ at time $t_k$ in a collection of $M$ assets $\mathcal{P} = \{1,2,\dots,M\}$, given the price at time $t_{k-1}$, is:
$$
\boxed{
\begin{align*}
S_{i}(t_{k}) &= S_{i}(t_{k-1})\cdot\exp\Biggl[\bar g_{i}\cdot\Delta{t} + \sqrt{\Delta{t}}\cdot\sum_{j\in\mathcal{P}}a_{ij}\cdot{Z_{j}}\Biggr]\quad{i\in\mathcal{P}}\quad\text{for}\quad{k=1,2,\dots}\\
\end{align*}}
$$
where $S_{i}(t_{k-1})$ is the share price of asset $i\in\mathcal{P}$ at time $t_{k-1}$, $\bar g_i$ is expected log growth, $C_{ii}$ is the $(i,i)$-th element of the GBM covariance rate $\mathbf C$, $\Delta{t} = t_{k} - t_{k-1}$ is the time-step size, $a_{ij}$ is the $(i,j)$-th element of the matrix $\mathbf A$, and $Z_{j}\sim\mathcal{N}(0,1)$ are independent standard normal random variables. The corresponding realized growth is $g_{i,t,\Delta t}=\bar g_i+(\mathbf A\mathbf Z)_i/\sqrt{\Delta t}$.

While MAGBM is a powerful model, it requires knowledge of the expected log-growth vector and the covariance rate $\mathbf C$, which can be challenging to estimate accurately. Furthermore, we have all the caveats of GBM, such as the constant drift and volatility assumptions, and the assumption of a normal distribution of returns.

Finally, suppose we wanted to use MAGBM to simulate the prices of a portfolio of assets. In that case, we need to know the fraction of the total portfolio value invested in each asset, which adds another layer of complexity and is our topic for today's lecture.

Let's illustrate this with an example.

> __Example__
> 
> [▶ Let's simulate a portfolio using MAGBM](CHEME-5660-L5b-Example-MAGBM-Portfolio-Fall-2026.ipynb). In this example, we simulate the price of a portfolio of assets using the MAGBM model. We'll assume we have a portfolio consisting of several stocks and ETFs, each with a specified weight in the portfolio. We'll use historical data to estimate the parameters of the MAGBM model and then simulate the portfolio's price over time using out-of-sample data.
___

## Modern Portfolio Theory (MPT)
Modern Portfolio Theory (MPT) is a framework for constructing investment portfolios that aims to maximize expected growth rate for a given level of risk or, equivalently, minimize risk for a given level of expected growth rate.

> __Reference:__ Modern Portfolio Theory was introduced by Harry Markowitz in the 1950s and has since become a foundational concept in finance. Markowitz was awarded the Nobel Prize in Economic Sciences in 1990 for his pioneering work in this area. The original publication is given here: [Portfolio Selection, The Journal of Finance, Vol. 7, No. 1 (Mar., 1952), pp. 77-91](https://www.jstor.org/stable/2975974). The Nobel Prize information is available here: [The Sveriges Riksbank Prize in Economic Sciences in Memory of Alfred Nobel 1990](https://www.nobelprize.org/prizes/economic-sciences/1990/markowitz/facts/).

Let's warm up to MPT by watching a short video that introduces the key concepts: [here](https://www.youtube.com/watch?v=VsMpw-qnPZY)

The key idea of MPT is to optimally balance risk and reward by diversifying investments across different assets. Let's dig into this argument by first exploring what we mean by risk and reward of a portfolio, starting with reward.

### Portfolio Reward
The reward of a portfolio is measured by its expected growth rate, which is the weighted average of the expected growth rates of the individual assets in the portfolio. 

Suppose we have a portfolio $\mathcal{P}$ consisting of $M$ assets. Let $w_i\in\mathbb{R}_{\geq 0}$ be the weight of asset $i$ in the portfolio (i.e., the dollar fraction of the total portfolio value invested in asset $i$), and let $\mathbb{E}[g_i]$ be the expected growth rate of asset $i$. Then, the expected growth rate of the portfolio, denoted as $\mathbb{E}[g_{\mathcal{P}}]$, is given by:
$$
\mathbb{E}[g_{\mathcal{P}}] = \sum_{i=1}^{M} w_i \mathbb{E}[g_i]\quad\Longrightarrow\;\mathbf{w}^{\top}\mathbb{E}[\mathbf{g}]
$$
where $M$ is the total number of assets in the portfolio, i.e., $|\mathcal{P}| = M$, the weight vector is $\mathbf{w}^{\top} = [w_1, w_2, \dots, w_M]$, the sum of weights is one, and $\mathbb{E}[\mathbf{g}] = [\mathbb{E}[g_1], \mathbb{E}[g_2], \dots, \mathbb{E}[g_M]]^{\top}$ is the vector of expected growth rates of the individual assets.

> __Growth Rate versus Return__  
> 
> We could instead write the portfolio reward using log returns $r_i=(\Delta t)g_i$. Let $g_{\star}=r_{\star}/\Delta t$. Then the expected portfolio growth rate is:
> $$
\begin{align*}
\mathbb{E}[g_{\mathcal{P}}] & = \sum_{i=1}^{M} w_i \mathbb{E}[g_i]\\
& = \sum_{i=1}^{M} w_i \mathbb{E}\Bigl[\frac{r_i}{\Delta{t}}\Bigr]\quad\Longrightarrow\text{pull out } \frac{1}{\Delta{t}} \\
& = \frac{1}{\Delta{t}} \sum_{i=1}^{M} w_i \mathbb{E}[r_i]\\
& = \left(\frac{1}{\Delta{t}}\right)\mathbf{w}^{\top}\mathbb{E}[\mathbf{r}]\quad\blacksquare    
\end{align*}$$
> The expected log return $\mathbf{w}^{\top}\mathbb{E}[\mathbf{r}]$ is dimensionless, while $\mathbf{w}^{\top}\mathbb{E}[\mathbf{g}]$ has units $[\text{time}]^{-1}$. They are related by $\mathbb{E}[r_{\mathcal P}]=(\Delta t)\mathbb{E}[g_{\mathcal P}]$.

### Portfolio Risk
The risk of a portfolio is (typically) measured by its variance (or standard deviation) of growth rates, which takes into account the variances of the individual assets as well as the covariances between them. The portfolio variance written in terms of growth rates is given by:
$$
\text{Var}(g_{\mathcal{P}}) = \sum_{i=1}^{M} \sum_{j=1}^{M} w_i w_j \text{Cov}(g_i, g_j)\quad\Longrightarrow\;\mathbf{w}^{\top}\mathbf{\Sigma}_{g}\mathbf{w}
$$
where $\text{Cov}(g_i, g_j)$ is the covariance between the growth rates of assets $i$ and $j$, and $\mathbf{\Sigma}_{g}$ is the covariance matrix of asset growth rates, defined as:
$$
\mathbf{\Sigma}_{g} =
\begin{bmatrix}
\text{Var}(g_1) & \text{Cov}(g_1, g_2) & \cdots & \text{Cov}(g_1, g_M) \\
\text{Cov}(g_2, g_1) & \text{Var}(g_2) & \cdots & \text{Cov}(g_2, g_M) \\
\vdots & \vdots & \ddots & \vdots \\
\text{Cov}(g_M, g_1) & \text{Cov}(g_M, g_2) & \cdots & \text{Var}(g_M)
\end{bmatrix}
$$

However, we could also write the portfolio risk with respect to the returns instead of the growth rates, i.e., $\text{Cov}(r_i, r_j)$ and $\text{Var}(r_p)$.

> __Covariance of Growth Rates versus Returns__  
> 
> The portfolio risk argument remains the same. Let $g_{\star} = r_{\star}/\Delta{t}$. Then, the variance of the portfolio of $M$ assets is given by:
> $$
\begin{align*}
\text{Var}(g_{\mathcal{P}}) & = \sum_{i=1}^{M} \sum_{j=1}^{M} w_i w_j \text{Cov}(g_i, g_j)\\
& = \sum_{i=1}^{M} \sum_{j=1}^{M} w_i w_j \text{Cov}\Bigl(\frac{r_i}{\Delta{t}}, \frac{r_j}{\Delta{t}}\Bigr)\quad\Longrightarrow\text{pull out } \frac{1}{(\Delta{t})^2} \\
& = \frac{1}{(\Delta{t})^2} \sum_{i=1}^{M} \sum_{j=1}^{M} w_i w_j \text{Cov}(r_i, r_j)\\
& = \left(\frac{1}{(\Delta{t})^2}\right)\mathbf{w}^{\top}\mathbf{\Sigma_r}\mathbf{w}\quad\blacksquare    
\end{align*}$$
> where $\mathbf{\Sigma_r}$ is the covariance matrix of asset returns. The portfolio variance $\mathbf{w}^{\top}\mathbf{\Sigma_r}\mathbf{w}$ is __dimensionless__, while the portfolio variance of growth rates has units of $[\text{time}]^{-2}$. We can convert between the two by multiplying or dividing by $(\Delta{t})^2$. 

For GBM simulation or conventional annualized-volatility reporting, define the covariance rate $\mathbf C=\mathbf\Sigma_r/\Delta t=(\Delta t)\mathbf\Sigma_g$. This is a third covariance object, not the variance of the observed growth rates:
$$
\boxed{
\begin{align*}
C_{\mathcal P}=\frac{\operatorname{Var}(r_{\mathcal P})}{\Delta t}=(\Delta t)\operatorname{Var}(g_{\mathcal P})\quad\blacksquare
\end{align*}
}
$$

For daily observations, $\Delta t=1/252$ implies $\Sigma_g=252C$. The growth-rate variance therefore looks much larger than the conventional annual covariance rate; this is the required unit conversion, not a covariance-estimation failure. A common covariance rescaling leaves minimum-variance weights unchanged when growth targets are scaled consistently, but the numerical risk and Sharpe-ratio horizon must still be labeled.

Ok, we have our risk and reward measures for a portfolio of $M$ assets. Now, let's see how we can use this information to construct an optimal portfolio that balances these two forces.
___

<div>
    <center>
        <img src="figs/Fig-MinVar-Portfolio-RA-Schematic.png" width="680"/>
    </center>
</div>


## Optimal Weights for a Risky Portfolio
The goal of MPT is to find the optimal weights $\mathbf{w}$ that minimize the portfolio risk for a given level of expected growth rate, or equivalently, maximize the expected growth rate for a given level of risk. This is typically done by solving a __constrained quadratic programming__ problem. 

### Problem
Let's consider the case when we have a portfolio $\mathcal{P}$ consisting of $M$ __risky assets__, i.e., only equity, ETFs (or potentially derivatives) but no fixed income assets. In this case, we can formulate the optimal weights optimization problem (written in terms of growth rate) as:

$$
\boxed{
\begin{align*}
\text{minimize}~\text{Var}(g_{\mathcal{P}}) &= \sum_{i\in\mathcal{P}}\sum_{j\in\mathcal{P}}w_{i}w_{j}\underbrace{\text{Cov}\left(g_{i},g_{j}\right)}_{= \sigma_{i}\sigma_{j}\rho_{ij}}\quad{\Longleftrightarrow\mathbf{w}^\top \mathbf{\Sigma}_{g} \mathbf{w}} \\
\text{subject to}~\mathbb{E}(g_{\mathcal{P}})& =  \sum_{i\in\mathcal{P}}w_{i}\;\mathbb{E}(g_{i})= g^{*}\quad\Longleftrightarrow\mathbf{w}^\top \mathbb{E}(\mathbf{g}) = g^{*} \\
\sum_{i\in\mathcal{P}}w_{i} & =  1 \\
w_{i} & \geq  0\quad\forall{i}\in\mathcal{P}
\end{align*}}
$$
The term $g^{*}$ is the target growth rate for portfolio $\mathcal{P}$ specified by the investor. The $w_{i}\geq{0}~\forall{i}\in\mathcal{P}$ and the summation-to-unity constraints forbid short selling (borrowing). If short selling (borrowing) is allowed, these constraints can be relaxed.

> __Frontier:__ To construct the efficient frontier, we systematically vary $g^{*}$ and solve the optimization problem for each target growth-rate level, generating the set of minimum variance portfolios that trace out the efficient frontier curve. Each point on the frontier is a different set of portfolio weights.

Let's solve this problem to get a feel for how it works using an example.

> [▶ Let's compute the optimal weights for a portfolio using the minimum variance portfolio approach](CHEME-5660-L5b-Example-Data-MinVar-Portfolio-Fall-2026.ipynb). In this example, we compute the optimal weights for a portfolio of assets using the minimum variance portfolio approach. We'll compare the performance of this portfolio to an equally weighted portfolio and an index fund, e.g., `SPY`.

___


## Net Present Value of a Portfolio
The wealth of the portfolio $\mathcal{P}$ at any time $t$ is given by:
$$
\begin{equation*}
W_{\mathcal{P}}(t) = \sum_{a\in\mathcal{P}}n_{a}(t)\;{S}_{a}(t)
\end{equation*}
$$
where $n_{a}(t)$ is the number of shares of asset $a\in\mathcal{P}$ held at $t$, and
$S_{a}(t)$ denotes the share price of asset $a\in\mathcal{P}$ at time $t$. Suppose we initially invest wealth $W_{\mathcal{P}}(0)$ (units: USD) in a portfolio $\mathcal{P}$ at time $t = 0$. Then, the net present value (NPV) of the portfolio $\mathcal{P}$ at time $T$ (written from our perspective) is given by:
$$
\begin{equation*}
\texttt{NPV}(\bar{r}, T) = \underbrace{-W_{\mathcal{P}}(0)}_{\text{initial investment}} + 
\underbrace{W_{\mathcal{P}}(T)\cdot\mathcal{D}^{-1}_{T,0}(\bar{r})}_{\text{present value at $t=0$}}
\end{equation*}
$$
where $\mathcal{D}^{-1}_{T,0}(\bar{r})$ is the (inverse) continuous discount factor, and $\bar{r}$ denotes the (constant) discount rate. Dividing by the initial wealth gives the scaled NPV of the portfolio $\mathcal{P}$:
$$
\boxed{
\begin{equation*}
\underbrace{\frac{\texttt{NPV}(\bar{r},T)}{W_{\mathcal{P}}(0)}}_{\text{fractional return}} = \left[\frac{W_{\mathcal{P}}(T)}{W_{\mathcal{P}}(0)}\right]\cdot\mathcal{D}^{-1}_{T,0}(\bar{r}) - 1
\end{equation*}\quad\blacksquare}
$$
Ok, we know the initial investment. However, how much is the portfolio worth at the end of holding period $T$? To answer this question, we need the allocation vector $\mathbf{w}$, and an estimate of the prices of the assets in the portfolio over the holding period $T$. This is where MAGBM comes into play.

We'll start there next time.
___

## Summary
This lecture extended single-asset modeling to portfolio analysis using MAGBM. We explored how covariance matrices capture asset correlations, applied Modern Portfolio Theory to balance risk and return, and demonstrated optimization techniques for calculating minimum variance portfolio weights. The examples showed practical implementation of these concepts using real market data.

> __Key Takeaways__
> 
> * **Portfolio Risk and Return Mathematics**: Modern Portfolio Theory provides precise formulations for the risk and return of a portfolio. The portfolio expected growth rate is defined as $\mathbf{w}^{\top}\mathbb{E}[\mathbf{g}]$ while the portfolio risk is $\mathbf{w}^{\top}\mathbf{\Sigma}_{g}\mathbf{w}$, with covariance matrices capturing both individual asset variances and cross-asset correlations that determine diversification benefits.
>
> * **Constrained Quadratic Optimization Framework**: The minimum variance portfolio problem is structured as a constrained quadratic programming problem with non-negativity constraints and minimum return requirements. The minimum variance formula provides a systematic approach to optimal asset allocation decisions.
> * **MAGBM Model Capabilities and Limitations**: While MAGBM enables multi-asset portfolio simulation through correlated random processes using covariance matrices, it inherits GBM assumptions of constant drift and volatility, normal return distributions, and faces parameter estimation challenges from historical data.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___